In [5]:
from app.data.dto.messenger.ResponsePayload import ResponsePayload, ResponsePayloadCollection
import uuid
from app.services.email_sender import EmailSender, MAIL
from app.handlers.navigation_handler import NavigationHandler
from app.services.internal_api.map_builder_api import MapBuilderApi
from app.services.template.telegram_template_service import TemplateService

from dotenv import load_dotenv

load_dotenv()
from app.services.db_service import DbService
db = DbService()
await db.init_pool()

template_service = TemplateService(db, NavigationHandler(db), MapBuilderApi("https://api.thebunkering.com", "https://api.thebunkering.com"))
s = EmailSender()
route, err = await db.get_route_by_id("1a3e33f5-04ff-4bfd-8d7d-3154ac5ab137")
user, err = await db.get_user_by_id("a9a13e49-d7bc-4bd6-8b7a-c523c3e60e75")

In [3]:
print("Всего точек кооординат ", len(route.data.departure_to_destination_coordinates))
route.data.departure_to_destination_coordinates

Всего точек кооординат  790


[Coordinates(latitude=59.916139, longitude=30.241585),
 Coordinates(latitude=59.91518649047792, longitude=30.240037522539744),
 Coordinates(latitude=59.914829129659, longitude=30.236946568524427),
 Coordinates(latitude=59.9147494143692, longitude=30.23179631213397),
 Coordinates(latitude=59.92189562050557, longitude=30.198659323775388),
 Coordinates(latitude=59.93626774806812, longitude=30.137535603448676),
 Coordinates(latitude=59.940956256688445, longitude=30.095599200132767),
 Coordinates(latitude=59.93596114636654, longitude=30.07285011382766),
 Coordinates(latitude=59.9311279255078, longitude=30.051393546712095),
 Coordinates(latitude=59.92645659411224, longitude=30.031229498786075),
 Coordinates(latitude=59.92324063401998, longitude=30.018128547021508),
 Coordinates(latitude=59.92148004523104, longitude=30.012090691418386),
 Coordinates(latitude=59.93170236143962, longitude=29.95402754466322),
 Coordinates(latitude=59.95390758264572, longitude=29.843939106756004),
 Coordinates(la

In [11]:
import base64
from app.data.dto.main.SeaPort import SeaPortDB
from app.data import emoji
from jinja2 import Environment, FileSystemLoader, select_autoescape
from datetime import datetime
from weasyprint import HTML
from io import BytesIO
import re

env = Environment(
    loader=FileSystemLoader("templates"),
    autoescape=select_autoescape(["html", "xml"])
)

def render_delivery_basis( port: SeaPortDB):
    icons = []

    if port.barge_status:
        icons.append("🚢")
    if port.truck_status:
        icons.append("🚚")
    if getattr(port, "pipe_status", False):
        icons.append("🛢️")

    return " ".join(icons) if icons else "—"

async def format_option2_email2_jinja(user, route):
    departure = route.data.port_selection.departure_candidate
    destination = route.data.port_selection.destination_candidate
    steps = [s for s in route.bunkering_steps if s.selected]
    images = []

    # User info
    user_db, err = await db.get_user_by_id(route.user_id)
    subject = f"BunkeringBot {user_db.first_name or ''} {user_db.last_name or ''} {user_db.telegram_user_name or ''}" if user_db else ""

    # Map image
    indexed = []
    if departure:
        indexed.append(departure.to_indexed2(emoji.ARROW_UP, "green", "medium", True))
    if destination:
        indexed.append(destination.to_indexed2(emoji.ARROW_DOWN, "red", "medium", True))
    for step in steps:

        indexed.append(step.port.to_indexed2(str(step.n), "blue", "medium", False))

    image_data, image_err = await template_service.map_image_api.render_map(
        route.data.departure_to_destination_coordinates,
        indexed
    )
    map_link = template_service.map_image_api.get_route_map_link(str(route.id))

    # CONVERT PNG BYTES TO BASE64
    if image_data and not image_err:
        # Check if it's already base64 string
        if isinstance(image_data, str) and image_data.startswith('data:image'):
            # Already base64
            images.append(image_data.split(',')[1] if ',' in image_data else image_data)
        elif isinstance(image_data, bytes):
            # Convert PNG bytes to base64
            base64_image = base64.b64encode(image_data).decode('utf-8')
            images.append(base64_image)
        else:
            print(f"Unexpected image data type: {type(image_data)}")
    # Voyage overview
    overview = {
        "user": user,
        "departure": departure.format_port().replace("\n", ""),
        "destination": destination.format_port().replace("\n", ""),
        "departure_date": route.departure_date,
        "average_speed": route.average_speed_kts,
        "vessel_name": route.vessel_name,
        "imo_number": route.imo_number,
        "map_link": map_link
    }

    # Build fuel table
    # First, collect all unique fuels from all steps
    all_fuels_set = {fuel for step in steps for fuel in step.fuel_info.keys()}

    # Define preferred order - VLS FO and MGO LS must go first
    preferred_order = ["VLS FO", "MGO LS"]

    # Sort fuels: first preferred ones in order, then others alphabetically
    all_fuels = []
    # Add preferred fuels first (if they exist in the data)
    for fuel in preferred_order:
        if fuel in all_fuels_set:
            all_fuels.append(fuel)

    # Add remaining fuels (excluding the ones already added) in alphabetical order
    remaining_fuels = sorted([f for f in all_fuels_set if f not in preferred_order])
    all_fuels.extend(remaining_fuels)

    table_rows = []
    for step in steps:
        p = step.port
        row = {
            "Port": p.format_port().replace("\n", ""),
            "ETA": step.eta_datetime.strftime("%B %d, %Y") if step.eta_datetime else emoji.LINE,
            "Port info": p.agent_contact_list or emoji.LINE,
            "Fuel delivery method": render_delivery_basis(p)
        }

        for fuel in all_fuels:
            info = step.fuel_info.get(fuel, {})
            qty = info.get("quantity") or 0
            price = info.get("fuel_price")
            row[fuel] = f"{qty} mt — ${price}" if price else f"{qty} mt — Price on request"
        table_rows.append(row)

    # Cost summary
    fuel_totals = {}
    total_cost = 0
    for step in steps:
        for fuel_name, info in step.fuel_info.items():
            qty = info.get("quantity") or 0
            price = info.get("fuel_price") or 0
            cost = qty * price
            total_cost += cost
            if fuel_name not in fuel_totals:
                fuel_totals[fuel_name] = {"qty": 0, "cost": 0}
            fuel_totals[fuel_name]["qty"] += qty
            fuel_totals[fuel_name]["cost"] += cost

    # Render Jinja2 template
    template = env.get_template("bunkering_report.html")
    html_content = template.render(
        images=images,
        overview=overview,
        table_rows=table_rows,
        table_columns=["Port", "ETA", "Port info", "Fuel delivery method"] + all_fuels,
        fuel_totals=fuel_totals,
        total_cost=total_cost,
        generation_date=datetime.now().strftime("%d %B %Y"),

    )

    return html_content, subject, images, image_data




def html_to_pdf(html: str) -> bytes:
    buffer = BytesIO()
    HTML(string=html).write_pdf(buffer)
    return buffer.getvalue()

In [12]:

html_content, subject, images, image_data = await format_option2_email2_jinja(user, route)
pdf_bytes = html_to_pdf(html_content)

sender = EmailSender()
ok, err = await sender.route_report(
    subject=subject,
    text=html_content,
    images=[image_data],
    pdf_bytes=pdf_bytes,
    to=MAIL
)

In [5]:
ok

True